In [1]:
import os            # work with files and folders
import json          # turn Python objects into text and back (JSON)
import time          # measure how long things take
import random        # generate fake/synthetic data
import sqlite3        # a tiny built-in SQL database (no install needed)
import shutil         # copy/remove folders (used for snapshots/cleanup)

random.seed(42)      # fix the random seed so results are the same every run

WORK = "scratch"                     # name of our scratch working folder
shutil.rmtree(WORK, ignore_errors=True)  # delete it if it already exists
os.makedirs(WORK, exist_ok=True)     # create a fresh, empty scratch folder
print("Scratch folder ready at:", os.path.abspath(WORK))  # confirm location

Scratch folder ready at: /content/scratch


1 · Three shapes of storage: object · block · file
From the slides:

Object — each file is an object (data + metadata + unique key) in a flat bucket; no real folders. Great for scale and static data.
Block — a raw disk split into fixed-size blocks; lowest latency; what databases sit on.
File — a familiar folder/path hierarchy many machines can share.
We'll model all three in plain Python so the differences become obvious.

In [2]:
# ---------- OBJECT STORAGE (flat bucket: key -> {data, metadata}) ----------
object_bucket = {}   # a dict acts like a bucket; the dict key is the object key

def put_object(key, data, content_type):      # store one object whole
    object_bucket[key] = {                     # the value bundles everything
        "data": data,                          # the actual bytes/text
        "metadata": {                          # metadata travels WITH the object
            "size": len(data),                 # size in characters
            "content_type": content_type,      # e.g. image/png, text/plain
        },
    }

put_object("photos/cat.txt", "a cute cat", "text/plain")   # 'folders' are fake:
put_object("photos/dog.txt", "a good dog", "text/plain")   # they are just key text

print("Bucket keys:", list(object_bucket.keys()))          # flat list of keys
print("One object:", object_bucket["photos/cat.txt"])      # data + metadata together
# Note: to change an object you replace the WHOLE thing (no in-place edits).

Bucket keys: ['photos/cat.txt', 'photos/dog.txt']
One object: {'data': 'a cute cat', 'metadata': {'size': 10, 'content_type': 'text/plain'}}


In [3]:
# ---------- BLOCK STORAGE (one disk = list of fixed-size blocks) ----------
BLOCK_SIZE = 4                       # each block holds 4 characters (toy size)
disk = ["" for _ in range(8)]        # a 'disk' = 8 empty fixed-size blocks

def write_blocks(text):              # split text into fixed blocks and store
    for i in range(0, len(text), BLOCK_SIZE):     # step through text 4 chars at a time
        block_index = i // BLOCK_SIZE             # which block number this piece is
        disk[block_index] = text[i:i+BLOCK_SIZE]  # write that 4-char chunk in place

write_blocks("HELLO-DATABASE")       # store this string across several blocks
print("Raw blocks on disk:", disk)   # see the fixed-size pieces
# Block storage allows IN-PLACE edits: change just one block, not the whole file.
disk[0] = "JELL"                     # overwrite only block 0
print("After editing block 0:", disk)


Raw blocks on disk: ['HELL', 'O-DA', 'TABA', 'SE', '', '', '', '']
After editing block 0: ['JELL', 'O-DA', 'TABA', 'SE', '', '', '', '']


In [4]:
# ---------- FILE STORAGE (real folders + paths, shared filesystem) ----------
file_root = os.path.join(WORK, "fileshare")        # our 'mounted' share folder
os.makedirs(os.path.join(file_root, "team"), exist_ok=True)  # real nested folder

path = os.path.join(file_root, "team", "notes.txt")  # a real hierarchical path
with open(path, "w") as f:           # open the file for writing
    f.write("shared project notes")  # write some content

print("File exists at path:", os.path.exists(path))   # real path, real file
print("Contents:", open(path).read())                 # read it back
# Many machines could 'mount' file_root and see the same folders/paths.


File exists at path: True
Contents: shared project notes
